# 03 — Gold Model | RHC Training Analytics

Este notebook transforma a camada Silver em um **modelo analítico orientado ao Qlik Sense**.

Modelo inicial:

- `dim_date`
- `dim_profile`
- `dim_exercise`
- `dim_program`
- `fact_workout_session`
- `fact_workout_exercise`
- `fact_body_measurement`
- `fact_program_exposure`
- `fact_exercise_set` — uma linha por série, derivada do JSON `reps`

Fluxo: `Silver → Modelo dimensional / métricas → Gold Parquet → Qlik Sense`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Configuração e Silver mais recente


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 200)

BASE_DIR = Path('/content/drive/MyDrive/rhc-training-analytics/data')
SILVER_ROOT = BASE_DIR / 'silver'
GOLD_ROOT = BASE_DIR / 'gold'
RUN_TS = datetime.now(timezone.utc)
RUN_ID = RUN_TS.strftime('%Y%m%dT%H%M%SZ')

silver_runs = sorted(SILVER_ROOT.glob('process_date=*/run_id=*'))
if not silver_runs:
    raise FileNotFoundError('Nenhuma execução Silver encontrada.')
SILVER_DIR = silver_runs[-1]
GOLD_DIR = GOLD_ROOT / f'process_date={RUN_TS:%Y-%m-%d}' / f'run_id={RUN_ID}'
GOLD_DIR.mkdir(parents=True, exist_ok=True)

print('Silver selecionada:', SILVER_DIR)
print('Gold destino:', GOLD_DIR)


In [ ]:
TABLES = [
    'profiles', 'body_measurements', 'workout_sessions', 'workout_exercises',
    'exercise_catalog', 'exercise_records', 'training_programs', 'program_phases',
    'program_sessions', 'program_exercises', 'program_enrollments',
    'program_exercise_exposures',
]
s = {t: pd.read_parquet(SILVER_DIR / f'{t}.parquet') for t in TABLES}
pd.DataFrame([{'table': t, 'rows': len(df)} for t, df in s.items()])


## 2. Dimensões


In [ ]:
# Perfil
profile_cols = [c for c in ['id','name','birth_date','gender','goal','is_active','created_at'] if c in s['profiles'].columns]
dim_profile = s['profiles'][profile_cols].copy().rename(columns={'id': 'profile_id'})

# Exercício
exercise_cols = [c for c in [
    'id','code','name','primary_muscle_group_id','movement_pattern_id',
    'exercise_category_id','is_active'
] if c in s['exercise_catalog'].columns]
dim_exercise = s['exercise_catalog'][exercise_cols].copy().rename(columns={'id': 'exercise_id'})

# Programa
program_cols = [c for c in ['id','slug','name','objective','duration_weeks','status','description'] if c in s['training_programs'].columns]
dim_program = s['training_programs'][program_cols].copy().rename(columns={'id': 'program_id'})

display(dim_profile.head())
display(dim_exercise.head())
display(dim_program.head())


## 3. Fato de sessões de treino


In [ ]:
ws = s['workout_sessions'].copy()
session_cols = [c for c in [
    'id','profile_id','workout_code','workout_date','gym_name','duration_minutes',
    'completion_percentage','total_volume','distance_meters','duration_seconds',
    'average_pace_seconds_per_km','activity_mode','started_at','completed_at','source_system'
] if c in ws.columns]
fact_workout_session = ws[session_cols].copy().rename(columns={'id':'workout_session_id'})

if 'workout_date' in fact_workout_session.columns:
    fact_workout_session['date_key'] = pd.to_datetime(fact_workout_session['workout_date']).dt.strftime('%Y%m%d').astype('Int64')
fact_workout_session['session_count'] = 1
fact_workout_session.head()


## 4. Fato de exercícios realizados


In [ ]:
we = s['workout_exercises'].copy()
exercise_fact_cols = [c for c in [
    'id','session_id','exercise_id','sort_order','exercise_name_snapshot',
    'muscle_group_snapshot','movement_pattern_snapshot','prescribed_sets',
    'prescribed_reps','actual_reps','weight','completed','notes'
] if c in we.columns]
fact_workout_exercise = we[exercise_fact_cols].copy().rename(columns={
    'id':'workout_exercise_id', 'session_id':'workout_session_id'
})

session_bridge = fact_workout_session[[c for c in ['workout_session_id','profile_id','workout_date','date_key'] if c in fact_workout_session.columns]]
fact_workout_exercise = fact_workout_exercise.merge(session_bridge, on='workout_session_id', how='left')
fact_workout_exercise['exercise_row_count'] = 1
fact_workout_exercise.head()


## 5. Fato de medidas corporais


In [ ]:
bm = s['body_measurements'].copy()
measurement_cols = [c for c in ['id','profile_id','measured_on','weight','waist','chest','arm','thigh','hip','notes'] if c in bm.columns]
fact_body_measurement = bm[measurement_cols].copy().rename(columns={'id':'measurement_id'})
fact_body_measurement['date_key'] = pd.to_datetime(fact_body_measurement['measured_on']).dt.strftime('%Y%m%d').astype('Int64')
fact_body_measurement['measurement_count'] = 1
fact_body_measurement.head()


## 6. Fato de exposições do programa

`program_exercise_exposures` registra execução/progressão de exercícios vinculados ao programa. O JSON `reps` será preservado aqui e também explodido na fato de séries.


In [ ]:
pe = s['program_exercise_exposures'].copy()
exposure_cols = [c for c in [
    'id','profile_id','enrollment_id','program_exercise_id','program_id','workout_session_id',
    'variation_exercise_id','variation_name_snapshot','load','reps','completed_sets',
    'observed_rpe','progression_action','suggested_load','suggestion_reason',
    'accepted_action','created_at'
] if c in pe.columns]
fact_program_exposure = pe[exposure_cols].copy().rename(columns={'id':'exposure_id'})

if 'workout_session_id' in fact_program_exposure.columns:
    fact_program_exposure = fact_program_exposure.merge(
        session_bridge, on='workout_session_id', how='left', suffixes=('','_session')
    )
fact_program_exposure['exposure_count'] = 1
fact_program_exposure.head()


## 7. Explodir `reps` para granularidade de série

Esta é uma das transformações centrais do projeto: um array JSON como `[12, 11, 10]` passa a representar três linhas analíticas, uma por série.


In [ ]:
def parse_reps(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        return list(value)
    if pd.isna(value):
        return []
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    return []

set_rows = []
for _, row in fact_program_exposure.iterrows():
    reps_list = parse_reps(row.get('reps'))
    for set_number, reps in enumerate(reps_list, start=1):
        record = {
            'exposure_id': row.get('exposure_id'),
            'profile_id': row.get('profile_id'),
            'program_id': row.get('program_id'),
            'program_exercise_id': row.get('program_exercise_id'),
            'workout_session_id': row.get('workout_session_id'),
            'variation_exercise_id': row.get('variation_exercise_id'),
            'workout_date': row.get('workout_date'),
            'date_key': row.get('date_key'),
            'set_number': set_number,
            'reps': pd.to_numeric(reps, errors='coerce'),
            'load': pd.to_numeric(row.get('load'), errors='coerce'),
            'observed_rpe': pd.to_numeric(row.get('observed_rpe'), errors='coerce'),
        }
        record['set_volume'] = record['load'] * record['reps'] if pd.notna(record['load']) and pd.notna(record['reps']) else np.nan
        set_rows.append(record)

fact_exercise_set = pd.DataFrame(set_rows)
fact_exercise_set['set_count'] = 1
display(fact_exercise_set.head(20))
print('Séries geradas:', len(fact_exercise_set))


## 8. Calendário analítico


In [ ]:
date_series = []
for df, col in [
    (fact_workout_session, 'workout_date'),
    (fact_body_measurement, 'measured_on'),
    (s['program_enrollments'], 'start_date'),
]:
    if col in df.columns:
        date_series.extend(pd.to_datetime(df[col], errors='coerce').dropna().tolist())

if not date_series:
    raise ValueError('Nenhuma data encontrada para construir dim_date.')

calendar = pd.date_range(min(date_series).normalize(), max(date_series).normalize(), freq='D')
dim_date = pd.DataFrame({'date': calendar})
dim_date['date_key'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['year'] = dim_date['date'].dt.year
dim_date['quarter'] = 'Q' + dim_date['date'].dt.quarter.astype(str)
dim_date['month'] = dim_date['date'].dt.month
dim_date['month_name'] = dim_date['date'].dt.month_name()
dim_date['year_month'] = dim_date['date'].dt.strftime('%Y-%m')
dim_date['week'] = dim_date['date'].dt.isocalendar().week.astype(int)
dim_date['day'] = dim_date['date'].dt.day
dim_date['weekday'] = dim_date['date'].dt.weekday + 1
dim_date['weekday_name'] = dim_date['date'].dt.day_name()
dim_date['is_weekend'] = dim_date['weekday'].isin([6,7])
dim_date.head()


## 9. Indicadores de reconciliação e qualidade Gold


In [ ]:
quality_checks = pd.DataFrame([
    {'check':'workout_sessions_row_count', 'expected':len(s['workout_sessions']), 'actual':len(fact_workout_session)},
    {'check':'workout_exercises_row_count', 'expected':len(s['workout_exercises']), 'actual':len(fact_workout_exercise)},
    {'check':'body_measurements_row_count', 'expected':len(s['body_measurements']), 'actual':len(fact_body_measurement)},
    {'check':'program_exposures_row_count', 'expected':len(s['program_exercise_exposures']), 'actual':len(fact_program_exposure)},
])
quality_checks['passed'] = quality_checks.expected == quality_checks.actual
quality_checks


## 10. Persistir Gold


In [ ]:
gold = {
    'dim_date': dim_date,
    'dim_profile': dim_profile,
    'dim_exercise': dim_exercise,
    'dim_program': dim_program,
    'fact_workout_session': fact_workout_session,
    'fact_workout_exercise': fact_workout_exercise,
    'fact_body_measurement': fact_body_measurement,
    'fact_program_exposure': fact_program_exposure,
    'fact_exercise_set': fact_exercise_set,
}

for name, df in gold.items():
    df.to_parquet(GOLD_DIR / f'{name}.parquet', index=False)
quality_checks.to_parquet(GOLD_DIR / '_quality_checks.parquet', index=False)

manifest = {
    'project': 'rhc-training-analytics',
    'layer': 'gold',
    'run_id': RUN_ID,
    'processed_at_utc': RUN_TS.isoformat(),
    'silver_source': str(SILVER_DIR),
    'gold_output': str(GOLD_DIR),
    'datasets': {name: len(df) for name, df in gold.items()},
    'consumer': 'Qlik Sense',
}
with (GOLD_DIR / '_manifest.json').open('w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

manifest


## 11. Resumo final


In [ ]:
assert quality_checks['passed'].all(), 'Falha de reconciliação Silver → Gold.'

summary = pd.DataFrame([
    {'dataset': name, 'rows': len(df), 'columns': len(df.columns)}
    for name, df in gold.items()
]).sort_values('dataset')
display(summary)

print('✅ Gold construída com sucesso.')
print('Diretório:', GOLD_DIR)
print('Próxima etapa: script de carga e modelo associativo no Qlik Sense.')
